# 3 - Download Census Data


**Variables I'm pulling (all from ACS 5-year, table family B):**

| Concept | Census table | How to compute |
|---|---|---|
| Median household income | `B19013_001E` | Direct |
| % bachelor's degree or higher | `B15003_022E..025E / B15003_001E` | Sum bach+master+prof+doc, divide by total 25+ |
| % below poverty | `B17001_002E / B17001_001E` | Direct |
| % non-Hispanic white | `B03002_003E / B03002_001E` | Direct |
| Median age | `B01002_001E` | Direct |
| Total population | `B01003_001E` | Direct (used for log-pop control + sanity check) |

In [ ]:
import requests
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()  

Key loaded: 71e9a3...a77d


In [2]:
VARS = [
    'B19013_001E',                                              # median HH income
    'B15003_001E','B15003_022E','B15003_023E','B15003_024E','B15003_025E',  # education
    'B17001_001E','B17001_002E',                                # poverty
    'B03002_001E','B03002_003E',                                # race (NH white)
    'B01002_001E',                                              # median age
    'B01003_001E',                                              # total population
]

url = 'https://api.census.gov/data/2022/acs/acs5'
params = {
    'get': 'NAME,' + ','.join(VARS),
    'for': 'county:*',
    'in':  'state:*',
    'key': KEY,
}
print('Calling Census ACS 2022 5-year API for all US counties...')
r = requests.get(url, params=params, timeout=60)
r.raise_for_status()
data = r.json()
print('Got', len(data) - 1, 'rows (counties + DC + PR if included)')

Calling Census ACS 2022 5-year API for all US counties...
Got 3222 rows (counties + DC + PR if included)


In [3]:
# Build dataframe (first row of API response is headers)
raw = pd.DataFrame(data[1:], columns=data[0])

# Numeric conversion (API returns everything as strings)
for v in VARS:
    raw[v] = pd.to_numeric(raw[v], errors='coerce')

# Build 5-digit FIPS
raw['fips'] = raw['state'].astype(str).str.zfill(2) + raw['county'].astype(str).str.zfill(3)
raw.head()

,NAME,B19013_001E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,B17001_001E,B17001_002E,B03002_001E,B03002_003E,B01002_001E,B01003_001E,state,county,fips
0,"Autauga County, Alabama",68315,40188,6726,4014,702,437,58291,6630,58761,42635,39.0,58761,01,001,01001
1,"Baldwin County, Alabama",71039,167022,33474,15077,3483,2351,229539,23445,233420,192161,43.7,233420,01,003,01003
2,"Barbour County, Alabama",39712,17675,1167,640,188,105,21851,5280,24877,11084,40.6,24877,01,005,01005
3,"Bibb County, Alabama",50669,15925,1047,507,109,76,20836,4297,22251,16520,40.3,22251,01,007,01007
4,"Blount County, Alabama",57440,40817,3840,1751,270,156,58399,8277,59077,50614,40.8,59077,01,009,01009


### Compute derived percentages

In [4]:
census = pd.DataFrame()
census['fips'] = raw['fips']
census['name'] = raw['NAME']
census['median_hh_income']     = raw['B19013_001E']
census['median_age']           = raw['B01002_001E']
census['population']           = raw['B01003_001E']
census['log_population']       = np.log10(raw['B01003_001E'].clip(lower=1))

# % bachelor's or higher (of pop 25+)
census['pct_bachelors_plus'] = (
    (raw['B15003_022E'] + raw['B15003_023E'] + raw['B15003_024E'] + raw['B15003_025E'])
    / raw['B15003_001E'] * 100
)
# % below poverty
census['pct_poverty'] = raw['B17001_002E'] / raw['B17001_001E'] * 100
# % non-Hispanic white
census['pct_nhwhite'] = raw['B03002_003E'] / raw['B03002_001E'] * 100

# Drop Puerto Rico (state FIPS 72) - CHR data is 50 states + DC only
census = census[~census['fips'].str.startswith('72')].reset_index(drop=True)
print('Census rows after dropping PR:', len(census))
census.head()

Census rows after dropping PR: 3144


,fips,name,median_hh_income,median_age,population,log_population,pct_bachelors_plus,pct_poverty,pct_nhwhite
0,01001,"Autauga County, Alabama",68315,39.0,58761,4.769089,29.558575,11.373969,72.556628
1,01003,"Baldwin County, Alabama",71039,43.7,233420,5.368138,32.561579,10.213951,82.324137
2,01005,"Barbour County, Alabama",39712,40.6,24877,4.395798,11.881188,24.163654,44.555212
3,01007,"Bibb County, Alabama",50669,40.3,22251,4.347350,10.919937,20.622960,74.243854
4,01009,"Blount County, Alabama",57440,40.8,59077,4.771418,14.741407,14.173188,85.674628


In [5]:
# Sanity-check distributions
summary_cols = ['median_hh_income','pct_bachelors_plus','pct_poverty','pct_nhwhite','median_age','population']
census[summary_cols].describe().round(1)

,median_hh_income,pct_bachelors_plus,pct_poverty,pct_nhwhite,median_age,population
count,3144.0,3144.0,3144.0,3144.0,3144.0,3144.0
mean,-148752.3,23.5,14.4,74.7,41.6,105310.9
std,11890747.3,10.1,6.0,20.2,5.4,333792.4
min,-666666666.0,0.0,1.6,1.7,21.6,50.0
25%,52544.5,16.5,10.0,62.8,38.5,10835.8
50%,60931.0,20.9,13.4,81.8,41.4,25784.5
75%,70605.2,27.9,17.5,91.0,44.5,68079.8
max,170463.0,78.9,55.8,100.0,68.3,9936690.0


In [6]:
os.makedirs('data', exist_ok=True)
census.to_csv('data/census_df.csv', index=False)
print('Wrote data/census_df.csv -', len(census), 'counties')

Wrote data/census_df.csv - 3144 counties
